In [ ]:
"""
=============================================================
FILE 31 — MULTI-AGENT DEBATE SYSTEM
=============================================================

CONCEPTS TAUGHT
----------------
1. Multi-Agent Debate
2. Collective Intelligence
3. Debate Agents
4. Critic Agents
5. Judge Agents
6. Consensus Building
7. Adversarial Reasoning
8. Multi-Perspective Thinking
9. AI Deliberation Systems
10. Collaborative Reasoning

CORE IDEA
-----------
Multiple agents debate a topic.
A judge evaluates the arguments.

FLOW
-----
Agent A Argument
        ↓
Agent B Counter Argument
        ↓
Judge Evaluation
        ↓
Final Decision

REAL WORLD USE CASES
---------------------
- Legal AI
- Strategic analysis
- AI alignment
- Research reasoning
- Policy analysis
"""

# ============================================================
# STEP 1 — IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

from IPython.display import Image, display

# ============================================================
# STEP 2 — LOAD ENV VARIABLES
# ============================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ============================================================
# STEP 3 — INITIALIZE LLM
# ============================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# ============================================================
# STEP 4 — DEFINE STATE
# ============================================================

class State(TypedDict):
    topic: str
    agent_a_argument: str
    agent_b_argument: str
    final_decision: str

# ============================================================
# STEP 5 — AGENT A
# ============================================================

def agent_a(state: State):

    """
    First debater.
    """

    print("\nAgent A debating...\n")

    response = llm.invoke(
        f"""
        Argue IN FAVOR of:

        {state['topic']}
        """
    )

    return {
        "agent_a_argument": response.content
    }

# ============================================================
# STEP 6 — AGENT B
# ============================================================

def agent_b(state: State):

    """
    Counter argument agent.
    """

    print("\nAgent B debating...\n")

    response = llm.invoke(
        f"""
        Argue AGAINST:

        {state['topic']}

        Counter this argument:
        {state['agent_a_argument']}
        """
    )

    return {
        "agent_b_argument": response.content
    }

# ============================================================
# STEP 7 — JUDGE AGENT
# ============================================================

def judge_agent(state: State):

    """
    Judge evaluates both arguments.
    """

    print("\nJudge evaluating debate...\n")

    response = llm.invoke(
        f"""
        Evaluate the debate below.

        AGENT A:
        {state['agent_a_argument']}

        AGENT B:
        {state['agent_b_argument']}

        Decide:
        - stronger argument
        - weaknesses
        - final conclusion
        """
    )

    return {
        "final_decision": response.content
    }

# ============================================================
# STEP 8 — BUILD GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node("agent_a", agent_a)

builder.add_node("agent_b", agent_b)

builder.add_node("judge_agent", judge_agent)

# ============================================================
# STEP 9 — DEFINE EDGES
# ============================================================

builder.add_edge(
    START,
    "agent_a"
)

builder.add_edge(
    "agent_a",
    "agent_b"
)

builder.add_edge(
    "agent_b",
    "judge_agent"
)

builder.add_edge(
    "judge_agent",
    END
)

# ============================================================
# STEP 10 — COMPILE GRAPH
# ============================================================

graph = builder.compile()

# ============================================================
# STEP 11 — VISUALIZE GRAPH
# ============================================================

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

# ============================================================
# STEP 12 — RUN WORKFLOW
# ============================================================

result = graph.invoke(
    {
        "topic":
        """
        Should AI replace human teachers?
        """
    }
)

# ============================================================
# STEP 13 — PRINT RESULTS
# ============================================================

print("\nAGENT A ARGUMENT\n")
print("=" * 60)
print(result["agent_a_argument"])

print("\nAGENT B ARGUMENT\n")
print("=" * 60)
print(result["agent_b_argument"])

print("\nFINAL JUDGE DECISION\n")
print("=" * 60)
print(result["final_decision"])